In [3]:
import os


In [4]:
%pwd

'd:\\Text-Summerizer-project\\research'

In [5]:
os.chdir("../")

In [6]:
%pwd

'd:\\Text-Summerizer-project'

In [7]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_name: Path


In [8]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [24]:
class  ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        
        create_directories([self.config.artifacts_root])
    
    def get_model_evaluation_config(self):

        config = self.config.model_evaluation
        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_path=config.model_path,
            tokenizer_path=config.tokenizer_path,
            metric_file_name=config.metric_file_name
        )

        return model_evaluation_config
        
        

In [26]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk, load_metric
import torch
import pandas as pd
from tqdm import tqdm

In [27]:
import os

print(os.getcwd())

D:\Text-Summerizer-project


In [28]:
import os

os.chdir(r"D:\Text-Summerizer-project")
print(os.getcwd())

D:\Text-Summerizer-project


In [29]:
from datasets import load_from_disk

dataset_samsum = load_from_disk(
    "artifacts/data_transformation/samsum_dataset"
)

In [30]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [31]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")

In [32]:
from transformers import AutoModelForSeq2SeqLM

model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-small"
)

In [33]:
import evaluate

metric = evaluate.load("rouge")

In [34]:
from datasets import load_from_disk

dataset_samsum = load_from_disk(
    r"D:\Text-Summerizer-project\artifacts\data_transformation\samsum_dataset"
)

In [35]:
import gc
import torch
from tqdm import tqdm
import evaluate
import pandas as pd
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


class ModelEvaluation:
    def __init__(self, config):
        self.config = config

    def generate_batch_sized_chunks(self, list_of_elements, batch_size):
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i:i + batch_size]

    def calculate_metric_on_test_ds(
        self,
        dataset,
        metric,
        model,
        tokenizer,
        batch_size=1,
        device="cpu",
        column_text="dialogue",
        column_summary="summary"
    ):

        model.eval()

        article_batches = list(
            self.generate_batch_sized_chunks(dataset[column_text], batch_size)
        )

        target_batches = list(
            self.generate_batch_sized_chunks(dataset[column_summary], batch_size)
        )

        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches),
            total=len(article_batches)
        ):

            inputs = tokenizer(
                article_batch,
                max_length=128,
                truncation=True,
                padding=True,
                return_tensors="pt"
            )

            # ✅ FIX: ensure SAME device as model
            inputs = {k: v.to(model.device) for k, v in inputs.items()}

            with torch.no_grad():
                summaries = model.generate(
                    **inputs,
                    max_length=32,
                    num_beams=1,
                    early_stopping=True
                )

            decoded_summaries = tokenizer.batch_decode(
                summaries,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True
            )

            metric.add_batch(
                predictions=decoded_summaries,
                references=target_batch
            )

            del inputs
            del summaries
            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        return metric.compute()

    def evaluate(self):

        device = "cuda" if torch.cuda.is_available() else "cpu"

        tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path)

        model = AutoModelForSeq2SeqLM.from_pretrained(
            self.config.model_path
        ).to(device)   # ✅ FIX: model moved to GPU

        dataset_samsum = load_from_disk(self.config.data_path)

        rouge_metric = evaluate.load("rouge")

        score = self.calculate_metric_on_test_ds(
            dataset=dataset_samsum["test"].select(range(10)),  # small test for speed
            metric=rouge_metric,
            model=model,
            tokenizer=tokenizer,
            batch_size=2,
            device=device,
            column_text="dialogue",
            column_summary="summary"
        )

        print(score)

        pd.DataFrame([score]).to_csv(
            self.config.metric_file_name,
            index=False
        )

        return score

In [36]:
config = ConfigurationManager()

model_eval_config = config.get_model_evaluation_config()

print(model_eval_config)
print(type(model_eval_config))

[2026-06-05 18:03:57,819:INFO:textSummarizerLogger:yaml file: config\config.yaml loaded successfully]
[2026-06-05 18:03:57,829:INFO:textSummarizerLogger:yaml file: params.yaml loaded successfully]
[2026-06-05 18:03:57,831:INFO:textSummarizerLogger:created directory at: artifacts]
[2026-06-05 18:03:57,833:INFO:textSummarizerLogger:created directory at: artifacts/model_evaluation]
ModelEvaluationConfig(root_dir='artifacts/model_evaluation', data_path='artifacts/data_transformation/samsum_dataset', model_path='artifacts/model_trainer/flan-t5-small-samsum-model', tokenizer_path='artifacts/model_trainer/tokenizer', metric_file_name='artifacts/model_evaluation/metrics.csv')
<class '__main__.ModelEvaluationConfig'>


In [37]:
import os
print(os.getcwd())

D:\Text-Summerizer-project


In [38]:
import os
os.chdir(r"D:\Text-Summerizer-project")
print(os.getcwd())

D:\Text-Summerizer-project


In [39]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.evaluate()
except Exception as e:
    raise e

[2026-06-05 18:04:00,762:INFO:textSummarizerLogger:yaml file: config\config.yaml loaded successfully]
[2026-06-05 18:04:00,764:INFO:textSummarizerLogger:yaml file: params.yaml loaded successfully]
[2026-06-05 18:04:00,765:INFO:textSummarizerLogger:created directory at: artifacts]
[2026-06-05 18:04:00,766:INFO:textSummarizerLogger:created directory at: artifacts/model_evaluation]


100%|██████████| 5/5 [00:03<00:00,  1.25it/s]

[2026-06-05 18:04:08,236:INFO:absl:Using default tokenizer.]
{'rouge1': np.float64(0.3793287047768557), 'rouge2': np.float64(0.12743023729022918), 'rougeL': np.float64(0.3068872362643795), 'rougeLsum': np.float64(0.304830355536301)}
